# Linear regression — from scratch

*Companion to **Eps 19–23** of the Linear Regression series.*

We build the simplest model in ML, one concept at a time. By the end you'll have trained it three ways: by hand, in PyTorch, in sklearn.

## Chapter 1 · Setup

Just the imports. NumPy for the math, matplotlib for plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Chapter 2 · The dataset

Seven cars. Each has a weight (in thousands of pounds) and an MPG. That's it.

In [ ]:
weights = np.array([2.4, 2.8, 3.0, 3.2, 3.6, 4.0, 4.4])
mpgs    = np.array([24,  22,  20,  19,  17,  15,  14])

Print it as a table so we can see the pattern.

In [ ]:
for w, m in zip(weights, mpgs):
    print(f'  {w} K lbs   ->   {m} MPG')

Heavier cars get fewer MPG. The relationship is visible in the numbers. Now let's plot it.

In [ ]:
plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3)
plt.xlabel('weight (1000 lbs)')
plt.ylabel('MPG')
plt.title('7 cars — weight vs MPG')
plt.show()

The dots roughly follow a downward line. **A line is the simplest model that could capture this pattern.**

We want to find *the* line — the best one. Let's build up to that.

## Chapter 3 · The model: `y = w·x + b`

A line has two numbers that define it:

- **`w`** is the slope — how steep the line is.
- **`b`** is the intercept — where the line crosses the y-axis.

Pick any (w, b), get a line. Change either, get a different line.

In [ ]:
def predict(w, b, x):
    return w * x + b

Let's pick a starting guess. We'll come back and improve it. For now: `w = -3`, `b = 30`.

In [ ]:
w_guess, b_guess = -3.0, 30.0

Predict the MPG of the first car (weight 2.4).

In [ ]:
predict(w_guess, b_guess, 2.4)

Compare that to the actual MPG (24). Off by a bit. Now do it for all 7 cars.

In [ ]:
predictions = predict(w_guess, b_guess, weights)
predictions

Side-by-side comparison:

In [ ]:
for x, actual, pred in zip(weights, mpgs, predictions):
    print(f'  weight={x}   actual={actual}   predicted={pred:.1f}')

Let's plot the line over the data and see how it looks.

In [ ]:
xs = np.linspace(2, 5, 50)
ys = predict(w_guess, b_guess, xs)

plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3, label='data')
plt.plot(xs, ys, color='#FFD27F', linewidth=2.5, label=f'guess: y = {w_guess}x + {b_guess}')
plt.xlabel('weight (1000 lbs)'); plt.ylabel('MPG'); plt.legend(); plt.show()

The line is roughly in the right zone but not great. We need a way to *measure* how not-great it is.

## Chapter 4 · How wrong is it?

For each car, the **error** is the predicted MPG minus the actual MPG.

In [ ]:
errors = predictions - mpgs
errors

Negative numbers mean we under-predicted. Positive means we over-predicted.

We want one number that summarizes how wrong the whole model is.

**Why not just average the errors?** Because positives and negatives cancel.

In [ ]:
errors.mean()

If the model is over-predicting some cars and under-predicting others by the same amount, the average error is zero — but the model is still bad.

**The fix:** square the errors first. Squaring makes everything positive. Big errors get punished extra (squaring 10 is 100).

In [ ]:
squared_errors = errors ** 2
squared_errors

Now average those. That's the **Mean Squared Error** — MSE — the standard loss for regression.

In [ ]:
mse_value = squared_errors.mean()
mse_value

Let's wrap it in a function so we can call it for any (w, b).

In [ ]:
def mse(w, b):
    preds = predict(w, b, weights)
    return np.mean((mpgs - preds) ** 2)

mse(w_guess, b_guess)

Same number. Now we can ask: for any line, how bad is it?

## Chapter 5 · MSE has a picture

Each squared error is literally the area of a square. Side length = the vertical distance from the line to the point. Area = that distance squared.

Plot the squares and the total gray area is the sum of squared errors. Divide by N for MSE.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=5, label='data')
ax.plot(xs, predict(w_guess, b_guess, xs), color='#FFD27F', linewidth=2.5, label='guess')

for x_d, y_d in zip(weights, mpgs):
    p = predict(w_guess, b_guess, x_d)
    side = abs(y_d - p)
    ax.add_patch(plt.Rectangle((x_d, min(y_d, p)), side, side,
                                color='#7C4DFF', alpha=0.25, linewidth=0))
    ax.plot([x_d, x_d], [y_d, p], color='#7C4DFF', alpha=0.6, linewidth=1)

ax.set_xlabel('weight (1000 lbs)'); ax.set_ylabel('MPG')
ax.set_title('Squared distances — total area = MSE × N')
ax.set_aspect('equal'); ax.legend(); plt.show()

The bigger the line's mistakes, the bigger the squares. Move the line, squares grow or shrink. **MSE going down = the squares getting smaller, on average.**

## Chapter 6 · Better guesses

Let's try a few different (w, b) pairs and see which one is smallest.

In [ ]:
for w, b in [(-3, 30), (-4, 32), (-5, 34), (-6, 36), (-5, 35)]:
    print(f'  w={w:>4}  b={b:>4}   MSE = {mse(w, b):.3f}')

Manually tuning gets tedious fast. With two knobs, you could keep going — but real models have millions of parameters.

We need a way to find the best (w, b) automatically. That's gradient descent. But first, let's see what we're trying to navigate.

## Chapter 7 · The loss surface

MSE is a function of two inputs (w, b) and outputs one number. **Two inputs, one output. That's a surface.**

Compute MSE on a grid of (w, b) values.

In [ ]:
w_range = np.linspace(-10, 2, 60)
b_range = np.linspace(15, 45, 60)
W, B = np.meshgrid(w_range, b_range)

loss_grid = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        loss_grid[i, j] = mse(W[i, j], B[i, j])

print('grid shape:', loss_grid.shape)
print('min MSE on grid:', loss_grid.min().round(3))

Now plot it as a 3D surface.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, loss_grid, cmap='viridis', alpha=0.8, edgecolor='none')
ax.set_xlabel('w (slope)'); ax.set_ylabel('b (intercept)'); ax.set_zlabel('MSE')
ax.set_title('The loss surface — a clean convex bowl')
ax.view_init(elev=30, azim=-60)
plt.show()

**One bottom. No local pits. No tricks.** For linear regression, the bowl is always this clean — what we call *convex*.

Gradient descent walks downhill on this bowl. The bottom is the answer.

## Chapter 8 · The gradient

The **gradient** is the slope of the bowl at the current point. It tells us which direction is *uphill*.

For MSE, the math works out to:

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i x_i(w x_i + b - y_i)$$
$$\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (w x_i + b - y_i)$$

In code:

In [ ]:
def grad(w, b):
    errs = predict(w, b, weights) - mpgs
    dw = 2 * np.mean(weights * errs)
    db = 2 * np.mean(errs)
    return dw, db

Compute the gradient at our guess (w=-3, b=30).

In [ ]:
dw, db = grad(w_guess, b_guess)
print(f'  dL/dw = {dw:.3f}')
print(f'  dL/db = {db:.3f}')

Both numbers are negative.

**Reading the gradient:** the gradient points *uphill*. Negative means uphill is in the negative direction. So *downhill* is in the *positive* direction — both `w` and `b` should increase to lower the loss.

But wait — we already saw that the true optimum has `w ≈ -5` (more negative, not less). The path to the optimum isn't straight — we'll see it curve.

## Chapter 9 · One step

Take a tiny step in the opposite direction of the gradient. The size of the step is the **learning rate** — usually written `η` (eta) or `lr`.

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

In [ ]:
lr = 0.01
w_new = w_guess - lr * dw
b_new = b_guess - lr * db

print(f'  before:  w={w_guess:.3f}  b={b_guess:.3f}  MSE={mse(w_guess, b_guess):.3f}')
print(f'  after:   w={w_new:.3f}  b={b_new:.3f}  MSE={mse(w_new, b_new):.3f}')

MSE went down. One step worked. Now do it lots of times.

## Chapter 10 · The full descent loop

Same step, in a loop. Store the history so we can plot it later.

In [ ]:
w, b = -1.0, 15.0   # bad starting point so the descent is visible
lr = 0.01

history = {'w': [w], 'b': [b], 'mse': [mse(w, b)]}

for step in range(10000):
    dw, db = grad(w, b)
    w -= lr * dw
    b -= lr * db
    history['w'].append(w)
    history['b'].append(b)
    history['mse'].append(mse(w, b))

print(f'final:  w={w:.4f}  b={b:.4f}  MSE={mse(w, b):.4f}')

**About 10,000 iterations to fully converge.** That feels like a lot for two parameters. The reason: the MPG features aren't *normalized* — they range from 2.4 to 4.4. Gradient descent struggles when features are on different scales than each other or than the bias.

In real ML, you'd normalize the inputs (subtract mean, divide by std) and converge in a few hundred steps. Or use a smarter optimizer like Adam (PyTorch section below).

But this is gradient descent in its rawest form. Same algorithm from calculus Ep 17.

Plot the loss curve to see the descent.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['mse'], color='#FFD27F', linewidth=2)
plt.xlabel('step'); plt.ylabel('MSE')
plt.title('Loss going down over 10,000 iterations')
plt.yscale('log')
plt.show()

Steep drop, then a long flattening tail. Classic convergence shape.

## Chapter 11 · The descent path on the bowl

Overlay the path on the loss surface from Chapter 7. The ball rolls down.

In [ ]:
fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, loss_grid, cmap='viridis', alpha=0.55, edgecolor='none')

# subsample for readability — 10000 points would be too dense
idx = np.linspace(0, len(history['w']) - 1, 100).astype(int)
path_w = [history['w'][i] for i in idx]
path_b = [history['b'][i] for i in idx]
path_mse = [history['mse'][i] for i in idx]

ax.plot(path_w, path_b, path_mse, color='#FFD27F', linewidth=2, marker='o', markersize=3)
ax.scatter([path_w[0]], [path_b[0]], [path_mse[0]],
           color='#FF4081', s=150, edgecolor='white', linewidth=1.5, label='start')
ax.scatter([path_w[-1]], [path_b[-1]], [path_mse[-1]],
           color='#00E676', s=150, edgecolor='white', linewidth=1.5, label='end')

ax.set_xlabel('w'); ax.set_ylabel('b'); ax.set_zlabel('MSE')
ax.set_title('Gradient descent on the loss surface')
ax.view_init(elev=30, azim=-60); ax.legend(); plt.show()

Pink dot is where we started. Green dot is where we ended up. The line connecting them is the descent path. **That path is what training looks like.**

## Chapter 12 · The fitted line

Plot the trained model over the data and compare to where we started.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3, label='data')
plt.plot(xs, predict(w_guess, b_guess, xs), color='#FF4081', linewidth=2,
         linestyle='--', label=f'starting guess', alpha=0.6)
plt.plot(xs, predict(w, b, xs), color='#FFD27F', linewidth=2.5,
         label=f'fitted: y = {w:.2f}x + {b:.2f}')
plt.xlabel('weight (1000 lbs)'); plt.ylabel('MPG'); plt.legend(); plt.show()

The yellow line is the trained model. Tighter fit through the data than the pink dashed starting guess. **By-hand linear regression — done.**

## Chapter 13 · The same thing in PyTorch — the 5-line loop

PyTorch handles the gradient for us. `loss.backward()` computes ∂L/∂w and ∂L/∂b automatically (chain rule, from calculus Ep 15-18). The five lines below are the same loop we just wrote by hand.

In [ ]:
import torch
import torch.nn as nn

Convert data to tensors. Shape `(N, 1)` for the linear layer.

In [ ]:
X = torch.tensor(weights, dtype=torch.float32).view(-1, 1)
y = torch.tensor(mpgs,    dtype=torch.float32).view(-1, 1)

print('X.shape:', X.shape)
print('y.shape:', y.shape)

Set up the model, optimizer, and loss.

In [ ]:
model = nn.Linear(1, 1)              # y = wx + b
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

Why **Adam** instead of vanilla SGD? Adam adapts the learning rate per parameter, which handles un-normalized features gracefully. SGD would also work but need 10,000+ iterations like our by-hand version. Adam takes a few thousand.

The five lines:

In [ ]:
for step in range(5000):
    pred = model(X)            # forward pass
    loss = loss_fn(pred, y)    # compute MSE
    loss.backward()            # chain rule
    optimizer.step()           # gradient descent
    optimizer.zero_grad()      # reset for next step

Pull out the trained parameters and compare.

In [ ]:
w_torch = model.weight.item()
b_torch = model.bias.item()

print(f'  PyTorch:   w={w_torch:.4f}   b={b_torch:.4f}')
print(f'  By-hand:   w={w:.4f}   b={b:.4f}')

Same answer (within numerical precision). Different code path, same destination.

## Chapter 14 · The same thing in sklearn — 3 lines

Sklearn wraps the whole training loop. For most projects, this is what you'd actually write.

In [ ]:
from sklearn.linear_model import LinearRegression

model_sk = LinearRegression()
model_sk.fit(weights.reshape(-1, 1), mpgs)

Read out the trained parameters.

In [ ]:
print(f'  sklearn:   w={model_sk.coef_[0]:.4f}   b={model_sk.intercept_:.4f}')
print(f'  PyTorch:   w={w_torch:.4f}   b={b_torch:.4f}')
print(f'  By-hand:   w={w:.4f}   b={b:.4f}')

All three agree.

Three different code paths, one true answer. The math doesn't care how you got there.

Predict the MPG of a brand-new 3,500-pound car.

In [ ]:
model_sk.predict([[3.5]])

About 17.9 MPG. Reasonable — it sits right in the middle of our training data range.

## Chapter 15 · Bonus — sklearn's algorithm roadmap

Sklearn ships with an [infamous algorithm cheat sheet](https://scikit-learn.org/stable/machine_learning_map.html) — answer a few questions about your data, it routes you to a starting model.

![sklearn algorithm map](https://scikit-learn.org/stable/_static/ml_map.svg)

**Walking the chart for our MPG problem:**

1. 50+ samples? *Pretend yes for the exercise (we actually only have 7).*
2. Predicting a category? **No** — we're predicting a number (MPG).
3. Predicting a quantity? **Yes** → regression branch.
4. Few features? **Yes** → `LinearRegression` ← we are here ✓

If we had 100,000+ samples, the chart routes us to `SGDRegressor`. With many noisy features, to `Lasso` or `Ridge` (regularized variants). Same `.fit(X, y)` interface — different sub-algorithm under the hood.

**The chart isn't gospel.** It's a sane starting point. The real workflow:

1. Pick a baseline from the chart.
2. Train it.
3. Look at the errors.
4. Iterate to a better model.

Linear regression is rarely the final model. But it's almost always a good first one — fast, interpretable, no hyperparameters to tune. If you can't beat it, you don't need anything fancier.

## Chapter 16 · More features

Real cars have more than just weight. Let's add horsepower and engine displacement.

In [ ]:
# Synthetic features correlated with weight, plus noise
horsepower   = weights * 60 + np.random.normal(0, 15, size=7)
displacement = weights * 1.5 + np.random.normal(0, 0.4, size=7)

X_multi = np.column_stack([weights, horsepower, displacement])
X_multi.shape

Three features per car instead of one. The model becomes:

$$y = b + w_1 \cdot \text{weight} + w_2 \cdot \text{horsepower} + w_3 \cdot \text{displacement}$$

*Same training loop. One extra weight per feature. That's it.*

In [ ]:
model_multi = LinearRegression()
model_multi.fit(X_multi, mpgs)

Read the coefficients — one per feature.

In [ ]:
print(f'  weight coef:        {model_multi.coef_[0]:+.4f}')
print(f'  horsepower coef:    {model_multi.coef_[1]:+.4f}')
print(f'  displacement coef:  {model_multi.coef_[2]:+.4f}')
print(f'  intercept:          {model_multi.intercept_:+.4f}')

Each coefficient says: **holding the other features constant, this is how much MPG changes per unit of this feature.**

All three are negative — heavier, more powerful, larger-engined cars get fewer MPG. Matches intuition.

## Chapter 17 · What we did

1. Loaded a dataset and plotted it.
2. Defined a model `y = wx + b` and a loss `MSE`.
3. Visualized MSE as squared distances, then as a 3D bowl over (w, b).
4. Computed the gradient by hand. Took one step. Looped 10,000 times.
5. Plotted the descent path on the bowl.
6. Did the same thing in PyTorch (5 lines) and sklearn (3 lines). All three converged to the same answer.
7. Extended to multiple features. Same loop. Same algorithm.

**Next notebook: logistic regression.** Same training loop with two substitutions — sigmoid in the forward pass, log loss in place of MSE.